# Day 28: Lab - SQL + Pandas Mini-Project

## Project Goal
This lab ties together everything from the SQL and pandas lessons into one mini end-to-end workflow, which mirrors what a real analytics task looks like: get raw data into a database, use SQL to answer business questions, then use pandas/matplotlib to visualize the results. We start by building a small `orders` dataset in pandas and loading it into an in-memory SQLite database with `to_sql`, the same pattern from Day 27. From here, every following step queries this `orders` table with SQL and pulls the results back into pandas for reporting - by the end you'll have gone from raw rows to a finished chart.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
# Small mock orders dataset: each row is one order, with the customer,
# product category, and amount spent - some customers appear multiple times
orders = pd.DataFrame({
    'order_id': range(1, 9),
    'customer': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob', 'Alice', 'Diana', 'Charlie'],
    'category': ['Electronics', 'Books', 'Electronics', 'Books', 'Electronics', 'Toys', 'Books', 'Toys'],
    'amount': [500, 20, 300, 15, 450, 60, 25, 80]
})
# Load the DataFrame into SQLite as an 'orders' table so we can query it with SQL
orders.to_sql('orders', conn, index=False, if_exists='replace')

## Step 1: Top Customers by Spend
A common business question is "who are our best customers?" - answering it requires grouping orders by customer and summing their total spend, which is exactly what SQL's `GROUP BY` is built for. `GROUP BY customer` collapses all rows for the same customer into one group, `SUM(amount)` adds up the amount within each group, and `ORDER BY total_spent DESC` sorts the result so the highest spenders appear first. The output is a small DataFrame with one row per customer and their total spend - reading top to bottom tells you immediately who to prioritize for retention or loyalty offers.

In [ ]:
# Group orders by customer, sum their spend, and sort highest spender first
query = '''
    SELECT customer, SUM(amount) AS total_spent
    FROM orders GROUP BY customer ORDER BY total_spent DESC
'''
print(pd.read_sql_query(query, conn))

## Step 2: Category Breakdown
Beyond individual customers, businesses also want to know which product categories drive the most revenue, so they can decide where to invest in marketing or stock. This query groups by `category` instead of `customer`, and computes two aggregates per group at once: `SUM(amount)` for total revenue and `COUNT(*)` for how many orders fell into that category. Sorting by `revenue DESC` puts the top-earning category first. Comparing `revenue` against `orders` per row is also useful - a category with fewer orders but high revenue (like Electronics here) indicates high-value purchases, while many orders with low revenue points to cheaper, high-volume items.

In [ ]:
# Group by category to get total revenue and order count per category
query = '''
    SELECT category, SUM(amount) AS revenue, COUNT(*) AS orders
    FROM orders GROUP BY category ORDER BY revenue DESC
'''
result = pd.read_sql_query(query, conn)
print(result)

## Step 3: Visualize
Numbers in a table are precise but hard to compare at a glance - a bar chart makes it immediate which category dominates revenue. Since `result` from the previous step is already a clean, aggregated DataFrame (one row per category), it can be plotted directly with pandas' built-in `.plot()` method without any extra reshaping. We plot `category` on the x-axis and `revenue` on the y-axis as bars, add a title, and label the y-axis so the chart is self-explanatory. Taller bars mean higher revenue - this is the visual endpoint of the whole SQL-to-pandas pipeline built in this lab.

In [ ]:
import matplotlib.pyplot as plt
# Plot the aggregated SQL result directly as a bar chart: one bar per category
result.plot(x='category', y='revenue', kind='bar', legend=False)
plt.title('Revenue by Category')
plt.ylabel('Revenue')
plt.show()